In [ ]:
# ─── Célula 1: Configuração ────────────────────────────────────────────────────
# Ajuste os caminhos caso execute fora do diretório raiz do projeto.
# Este notebook roda LOCALMENTE (CPU) — sem necessidade de GPU ou Google Colab.
#
# Seção da dissertação: 2.3.1 — CRF Baseline

import os

# ── Raiz do projeto ────────────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))

# ── Caminhos dos dados anotados (CoNLL) ───────────────────────────────────────
TRAIN_PATH = os.path.join(PROJECT_ROOT, 'outputs', 'ner', 'train.conll')
DEV_PATH   = os.path.join(PROJECT_ROOT, 'outputs', 'ner', 'dev.conll')
TEST_PATH  = os.path.join(PROJECT_ROOT, 'outputs', 'ner', 'test.conll')

# ── Saída do modelo treinado ───────────────────────────────────────────────────
MODEL_DIR  = os.path.join(PROJECT_ROOT, 'outputs', 'ner')
MODEL_PATH = os.path.join(MODEL_DIR, 'crf_model.joblib')

# ── Hiperparâmetros do CRF ────────────────────────────────────────────────────
C1             = 0.1    # regularização L1 (esparsidade)
C2             = 0.1    # regularização L2 (penaliza pesos grandes)
MAX_ITERATIONS = 300    # limite L-BFGS; 300 garante convergência neste corpus

# ── Adiciona src ao path para importar os módulos do projeto ──────────────────
import sys
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print('=' * 60)
print('  AnonClin — CRF Baseline (Seção 2.3.1)')
print('=' * 60)
print(f'  Train : {TRAIN_PATH}')
print(f'  Dev   : {DEV_PATH}')
print(f'  Test  : {TEST_PATH}')
print(f'  Modelo: {MODEL_PATH}')
print(f'  c1={C1}  c2={C2}  max_iter={MAX_ITERATIONS}')
print('=' * 60)

In [ ]:
# ─── Célula 2: Verificação de dependências ─────────────────────────────────────
# Instala pacotes caso não estejam presentes.
# Em ambiente virtual já configurado, esta célula pode ser pulada.

import importlib

REQUIRED = {
    'sklearn_crfsuite': 'sklearn-crfsuite',
    'seqeval'         : 'seqeval',
    'joblib'          : 'joblib',
}

missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]

if missing:
    print(f'Instalando: {missing}')
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + missing, check=True)
    print('Instalação concluída.')
else:
    print('✅  Todas as dependências já estão instaladas.')

import sklearn_crfsuite
import joblib
from seqeval.metrics import classification_report, f1_score

print(f'sklearn_crfsuite : {sklearn_crfsuite.__version__}')
print(f'joblib           : {joblib.__version__}')

In [ ]:
# ─── Célula 3: Leitura dos arquivos CoNLL ──────────────────────────────────────
# Formato esperado: token \t label por linha; linhas em branco separam sentenças.

def read_conll(path):
    """Lê arquivo CoNLL e retorna listas de tokens e labels por sentença."""
    sentences_tokens, sentences_labels = [], []
    tokens_cur, labels_cur = [], []

    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if line == '':
                if tokens_cur:
                    sentences_tokens.append(tokens_cur)
                    sentences_labels.append(labels_cur)
                    tokens_cur, labels_cur = [], []
            else:
                parts = line.split('\t')
                tokens_cur.append(parts[0])
                labels_cur.append(parts[1])

    if tokens_cur:   # última sentença sem linha em branco final
        sentences_tokens.append(tokens_cur)
        sentences_labels.append(labels_cur)

    return sentences_tokens, sentences_labels

# ── Carrega splits ─────────────────────────────────────────────────────────────
train_tokens, train_labels = read_conll(TRAIN_PATH)
dev_tokens,   dev_labels   = read_conll(DEV_PATH)
test_tokens,  test_labels  = read_conll(TEST_PATH)

print(f'Train : {len(train_tokens):>5} sentenças')
print(f'Dev   : {len(dev_tokens):>5} sentenças')
print(f'Test  : {len(test_tokens):>5} sentenças')

# Exemplo da primeira sentença de treino
print()
print('Exemplo (train[0]):')
for tok, lbl in zip(train_tokens[0], train_labels[0]):
    print(f'  {tok:<20} {lbl}')

In [ ]:
# ─── Célula 4: Extração de features ───────────────────────────────────────────
# Usa extrair_features_sentenca do módulo de pré-processamento do projeto.
# Features: sufixo/prefixo, maiúsculas, is_abrev, is_placeholder, contexto ±2.

import time
from preprocessamento.services.preprocessamento import extrair_features_sentenca

print('Extraindo features...')
t0 = time.time()

X_train = [extrair_features_sentenca(s) for s in train_tokens]
X_dev   = [extrair_features_sentenca(s) for s in dev_tokens]
X_test  = [extrair_features_sentenca(s) for s in test_tokens]

elapsed = time.time() - t0
print(f'Extração concluída em {elapsed:.1f}s')
print(f'Exemplo de features (token[0] da sentença[0]):')
for k, v in list(X_train[0][0].items())[:10]:
    print(f'  {k:<25} = {v}')

In [ ]:
# ─── Célula 5: Treinamento ─────────────────────────────────────────────────────
# Algoritmo: L-BFGS com regularização L1+L2.
# Convergência monitorada via training_log_ após o fit.
#
# ⚠️  Tempo estimado: 2–4 minutos dependendo da CPU.

import time
import os

print(f'Treinando CRF (c1={C1}, c2={C2}, max_iterations={MAX_ITERATIONS})...')
t0 = time.time()

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=C1,
    c2=C2,
    max_iterations=MAX_ITERATIONS,
    all_possible_transitions=True,   # aprende transições não vistas no treino
)
crf.fit(X_train, train_labels)

elapsed = time.time() - t0
print(f'Treino concluído em {elapsed:.1f}s')

# ── Diagnóstico de convergência ────────────────────────────────────────────────
log      = crf.training_log_
n_iter   = len(log.iterations)
last_it  = log.last_iteration

print()
print(f'Iterações realizadas : {n_iter} / {MAX_ITERATIONS}')
print(f'Convergiu?           : {"SIM" if n_iter < MAX_ITERATIONS else "NÃO (atingiu limite)"}')
print(f'Loss (última iter.)  : {last_it["loss"]:.4f}')
print(f'Error norm           : {last_it["error_norm"]:.5f}')

# ── Salva o modelo ────────────────────────────────────────────────────────────
os.makedirs(MODEL_DIR, exist_ok=True)
import joblib
joblib.dump(crf, MODEL_PATH)
print(f'\nModelo salvo em: {MODEL_PATH}')

In [ ]:
# ─── Célula 6: Avaliação no conjunto de teste ──────────────────────────────────
# Métrica principal: F1 entity-level com seqeval (mesmo método dos modelos BERT).
# Permite comparação direta na Tabela de Resultados da dissertação.

from seqeval.metrics import classification_report, f1_score

# ── Predição ──────────────────────────────────────────────────────────────────
y_pred_test = crf.predict(X_test)

# ── Métricas entity-level ──────────────────────────────────────────────────────
f1   = f1_score(test_labels, y_pred_test)
report = classification_report(test_labels, y_pred_test, digits=4)

print('=' * 60)
print(f'  CRF — Avaliação no Test Set (seqeval, entity-level)')
print('=' * 60)
print(report)
print(f'  micro-F1: {f1:.4f}')

# ── Avaliação no Dev (referência durante desenvolvimento) ─────────────────────
print()
print('─' * 60)
y_pred_dev = crf.predict(X_dev)
f1_dev = f1_score(dev_labels, y_pred_dev)
print(f'  Dev F1: {f1_dev:.4f}  (referência — não usada na comparação final)')
print('─' * 60)

In [ ]:
# ─── Célula 7: Features mais relevantes aprendidas pelo CRF ───────────────────
# Inspeciona os pesos de transição e os top features por classe.
# Útil para análise qualitativa na dissertação (Seção 4.x).

from collections import Counter

print('── Top 10 features positivas por classe ──')
for label in sorted(crf.classes_):
    if label.startswith('B-') or label == 'O':
        top_features = Counter()
        for attr, weights in crf.state_features_.items():
            if label in weights:
                top_features[attr[1]] = weights[label]
        top = top_features.most_common(5)
        if top:
            print(f'\n{label}:')
            for feat, w in top:
                print(f'  {feat:<40} {w:+.4f}')

print()
print('── Top 10 transições aprendidas ──')
top_trans = sorted(
    crf.transition_features_.items(),
    key=lambda x: x[1], reverse=True
)[:10]
for (from_l, to_l), weight in top_trans:
    print(f'  {from_l:<20} → {to_l:<20} {weight:+.4f}')

In [ ]:
# ─── Célula 8: Salvar resultado em JSON ───────────────────────────────────────
# Persiste as métricas para uso na Tabela de Resultados e nos gráficos.
# Mesmo formato que os resultados dos modelos BERT.

import json, datetime

RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'outputs', 'results')
RESULTS_FILE = os.path.join(RESULTS_DIR, 'crf_results.json')

os.makedirs(RESULTS_DIR, exist_ok=True)

# Extrai métricas por entidade do classification_report
from seqeval.metrics import classification_report as cr
import re

def parse_report(report_str):
    """Converte o classification_report do seqeval em dicionário."""
    result = {}
    for line in report_str.strip().split('\n'):
        parts = line.split()
        if len(parts) >= 5 and parts[0] not in ('precision', 'micro', 'macro', 'weighted'):
            label = parts[0]
            try:
                result[label] = {
                    'precision': float(parts[1]),
                    'recall'   : float(parts[2]),
                    'f1'       : float(parts[3]),
                    'support'  : int(parts[4]),
                }
            except ValueError:
                pass
    return result

results = {
    'model'         : 'CRF (sklearn-crfsuite)',
    'section'       : '2.3.1',
    'date'          : datetime.date.today().isoformat(),
    'hyperparams'   : {'c1': C1, 'c2': C2, 'max_iterations': MAX_ITERATIONS},
    'converged'     : len(log.iterations) < MAX_ITERATIONS,
    'n_iterations'  : len(log.iterations),
    'last_loss'     : last_it['loss'],
    'last_error_norm': last_it['error_norm'],
    'test_f1_micro' : round(f1, 4),
    'dev_f1_micro'  : round(f1_dev, 4),
    'per_entity'    : parse_report(report),
    'evaluation'    : 'seqeval entity-level micro-F1',
}

with open(RESULTS_FILE, 'w', encoding='utf-8') as fp:
    json.dump(results, fp, ensure_ascii=False, indent=2)

print(f'Resultados salvos em: {RESULTS_FILE}')
print()
print('Resumo:')
print(f'  Modelo        : {results["model"]}')
print(f'  Test micro-F1 : {results["test_f1_micro"]}')
print(f'  Dev  micro-F1 : {results["dev_f1_micro"]}')
print(f'  Convergiu?    : {results["converged"]}')
print(f'  Iterações     : {results["n_iterations"]} / {MAX_ITERATIONS}')